In [1]:
import os
import json
import ast
import re
import sys
import uuid
from typing import List
import shutil
import sqlite3

import inspect

from pathlib import Path
import importlib.util

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from IPython.display import Markdown, display


## Print one field of a table for a specific chart

In this example subtitle of a chart. 

In [ ]:
# import sqlite3
# import pandas as pd

# # Choose the chart_id you want
# chart_id = "016934e7c13b1b546fdea80efd074e02"

# # Connect to database
# conn = sqlite3.connect("../chart_database.db")

# query = """
# SELECT subtitle
# FROM chart
# WHERE id = ?
# """

# df = pd.read_sql_query(query, conn, params=[chart_id])

# conn.close()

# print(df['subtitle'][0])


## Generate a specific prompt text for one chart

In [3]:
# # 1) Add the *project root* (the parent of "src") to sys.path
# current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
# src_path = os.path.abspath(os.path.join(current_dir, '..', 'src'))
# sys.path.append(src_path)

# # 2) Import from the package path
# from b_func_prompt_texts import * 

In [4]:
# def get_prompt_data_from_chart(chart_id, conn):
#     chart_query = """
#     SELECT c.complex,
#            c.title, c.subtitle, c.notes,
#            c.x_axis_min, c.x_axis_max,
#            c.y_axis_min, c.y_axis_max,
#            ct.type AS chart_type
#     FROM chart c
#     JOIN chart_type ct ON c.chart_type_id = ct.id
#     WHERE c.id = ?
#     """
#     data_query = "SELECT DISTINCT x_category, y_category FROM chart_data WHERE chart_id = ?"
#     prognosis_query = "SELECT EXISTS (SELECT 1 FROM chart_data WHERE chart_id = ? AND prognosis = 1) AS has_prognosis;"
#     highlighted_query = "SELECT EXISTS (SELECT 1 FROM chart_data WHERE chart_id = ? AND highlighted = 1) AS is_highlighted;"
#     event_query = "SELECT type, date, label FROM data_event WHERE chart_id = ?"

#     chart_df = pd.read_sql_query(chart_query, conn, params=(chart_id,))
#     if chart_df.empty:
#         raise ValueError(f"No chart data found for chart_id={chart_id}")

#     data_df = pd.read_sql_query(data_query, conn, params=(chart_id,))
#     prognosis_df = pd.read_sql_query(prognosis_query, conn, params=(chart_id,))
#     highlighted_df = pd.read_sql_query(highlighted_query, conn, params=(chart_id,))
#     events_df = pd.read_sql_query(event_query, conn, params=(chart_id,))

#     def safe_val(val):
#         return str(val) if pd.notna(val) else None

#     row = chart_df.iloc[0]

#     return {
#         "complex": safe_val(row["complex"]),
#         "title": safe_val(row["title"]),
#         "subtitle": safe_val(row["subtitle"]),
#         "notes": safe_val(row["notes"]),
#         "chart_type": safe_val(row["chart_type"]),
#         "x_category": safe_val(data_df.iloc[0]["x_category"]),
#         "y_category": safe_val(data_df.iloc[0]["y_category"]),
#         "x_axis_min": safe_val(row["x_axis_min"]),
#         "x_axis_max": safe_val(row["x_axis_max"]),
#         "y_axis_min": safe_val(row["y_axis_min"]),
#         "y_axis_max": safe_val(row["y_axis_max"]),
#         "has_prognosis": bool(prognosis_df.iloc[0]["has_prognosis"]),
#         "is_highlighted": bool(highlighted_df.iloc[0]["is_highlighted"]),
#         "data_events": events_df.to_dict(orient="records") if not events_df.empty else np.nan
#     }

In [5]:
# chart_id = "016934e7c13b1b546fdea80efd074e02"

In [ ]:
conn = sqlite3.connect("../chart_database.db")

# img_path = os.path.join('data', 'NZZ_original', f'{chart_id}', f'{chart_id}.png')
# csv_path = os.path.join('..', 'data', 'data_nzz_csv', f'{chart_id}.csv')

# chart_info = get_prompt_data_from_chart(chart_id, conn)

# prompt_alt_text = alt_prompt_text_1(chart_info, csv_path, img_path)

# display(Markdown(f"**Prompt-Text**:\n```{prompt_alt_text}\n```"))

# conn.close()

In [ ]:
# SQLite-Verbindung erstellen
conn = sqlite3.connect("../chart_database.db")
cursor = conn.cursor()

# Tabellenliste
dbs = ["model", "chart_type", "chart", "chart_data", "data_event", "prompt_text_function", "alt_text_embedding", "alt_text", "generation_run"] #"gold_summary"

for db in dbs:
    print(f"\n------ Tabelle: {db} ------")
    cursor.execute(f"SELECT * FROM {db} LIMIT 5;")
    data = cursor.fetchall()

    # Spaltennamen anzeigen
    column_names = [description[0] for description in cursor.description]
    print(" | ".join(column_names))  # schöne Formatierung

    # Daten anzeigen
    for row in data:
        print(" | ".join(str(value) for value in row))

# Verbindung schließen
conn.close()


------ Tabelle: model ------
id | company | model_series | URL
1 | Meta | meta-llama/llama-4-maverick:free | https://openrouter.ai/meta-llama/llama-4-maverick:free
2 | google | google/gemini-2.5-flash-preview-09-2025 | https://statics.teams.cdn.office.net/evergreen-assets/safelinks/2/atp-safelinks.html
3 | OpenAI | openai/o4-mini | https://openrouter.ai/openai/o4-mini
4 | SentenceTransformers | sentence-transformers/all-MiniLM-L6-v2 | https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

------ Tabelle: chart_type ------
id | type
0 | Bar
1 | Line
2 | StackedBar
3 | Dotplot
4 | Arrow

------ Tabelle: chart ------
id | chart_type_id | x_axis_type | y_axis_type | x_axis_min | x_axis_max | y_axis_min | y_axis_max | complex | title | subtitle | notes
0023e8fed9d111fed753bb3f6b0afe78 | 0 | numeric | numeric | 2012.0 | 2017.0 | 5.01155899 | 20.79092879 | 1 | Auch der Handel mit der EU mit Autoteilen ist aus Sicht der USA defizitär | in Mrd. $ | None
0057a266a9c555637dabccece11c5468

In [ ]:
conn = sqlite3.connect("../chart_database.db")

query_prompt_text_function = "SELECT * FROM prompt_text_function"

df_prompt_text_function = pd.read_sql_query(query_prompt_text_function, conn)

conn.close()
df_prompt_text_function

## Model table
### Add new model

In [ ]:
# # Open DB connection
# conn = sqlite3.connect("../chart_database.db")
# cursor = conn.cursor()

# # New data — must be a list of tuples
# new_model_data = [
#     ("Meta", 
#      "meta-llama/llama-4-maverick", 
#      "05.04.2025", 
#      "https://openrouter.ai/meta-llama/llama-4-maverick")
# ]

# # Query
# insert_new_model_query = """
# INSERT INTO model (company, model_series, created, URL)
# VALUES (?, ?, ?, ?)
# """

# # Execute query
# cursor.executemany(insert_new_model_query, new_model_data)

# conn.commit()
# conn.close()

### Delete model

In [ ]:
# # Connect to the database
# conn = sqlite3.connect("../chart_database.db")
# cursor = conn.cursor()

# # ID of the row to delete
# target_id = 3

# # Delete query
# delete_query = "DELETE FROM model WHERE id = ?"

# # Execute deletion
# cursor.execute(delete_query, (target_id,))

# # Commit and close
# conn.commit()
# conn.close()

### Check model table

In [ ]:
# Check if insert was successfull
conn = sqlite3.connect("../chart_database.db")

query_model = "SELECT * FROM model"

df_model = pd.read_sql_query(query_model, conn)

conn.close()

df_model

,id,company,model_series,URL
0,1,Meta,meta-llama/llama-4-maverick:free,https://openrouter.ai/meta-llama/llama-4-maver...
1,2,google,google/gemini-2.5-flash-preview-09-2025,https://statics.teams.cdn.office.net/evergreen...
2,3,OpenAI,openai/o4-mini,https://openrouter.ai/openai/o4-mini
3,4,SentenceTransformers,sentence-transformers/all-MiniLM-L6-v2,https://huggingface.co/sentence-transformers/a...


## Prompt text function table

### Insert

In [ ]:
# Module path and name
file_path = "b3_prompt_text_functions.py"
module_name = "b3_prompt_text_functions"

# Dynamically import the module
spec = importlib.util.spec_from_file_location(module_name, file_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

func_name = "alt_prompt_text_8"

# Get the function object
func_obj = getattr(module, func_name)

# Get the source code
source = inspect.getsource(func_obj)

In [ ]:
# Open DB connection
conn = sqlite3.connect("../chart_database.db")
cursor = conn.cursor()

# Execute query
cursor.execute(
    "INSERT INTO prompt_text_function (name, content) VALUES (?, ?)",
    (func_name, source)
)

conn.commit()
conn.close()

### Check prompt text function table

In [ ]:
# Check if insert was successfull
conn = sqlite3.connect("../chart_database.db")

query_prompt_text_function = "SELECT * FROM prompt_text_function"

df_prompt_text_function = pd.read_sql_query(query_prompt_text_function, conn)

conn.close()
df_prompt_text_function

### Update content of already inserted prompt text function

In [ ]:
import sqlite3
import importlib.util
import inspect

# Module path and name
file_path = "b3_prompt_text_functions.py"
module_name = "b3_prompt_text_functions"

# Dynamically import the module
spec = importlib.util.spec_from_file_location(module_name, file_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

# List of functions you want to update
function_names = [
    "alt_prompt_text_6"
]

# Open DB connection
conn = sqlite3.connect("../chart_database.db")
cursor = conn.cursor()

# Loop through function names, get source code, and update content
for func_name in function_names:
    try:
        func_obj = getattr(module, func_name)
        source = inspect.getsource(func_obj)

        cursor.execute(
            "UPDATE prompt_text_function SET content = ? WHERE name = ?",
            (source, func_name)
        )

        print(f"Updated content for: {func_name}")
    except AttributeError:
        print(f"Function '{func_name}' not found in module.")

# Commit and close
conn.commit()
conn.close()

### Check alt text table

In [ ]:
# SQLite-Verbindung erstellen
conn = sqlite3.connect("../chart_database.db")
cursor = conn.cursor()

cursor.execute(f"SELECT * FROM generation_run;")
data = cursor.fetchall()

# Spaltennamen anzeigen
column_names = [description[0] for description in cursor.description]
print(" | ".join(column_names))  # schöne Formatierung

# Daten anzeigen
for row in data:
    print(" | ".join(str(value) for value in row))

# Verbindung schließen
conn.close()

id | chart_id | model_id | prompt_text_function_id | created_at | temperature | n_variants
1 | 006f6723bd41797d2bb1e3f65444757f | 2 | 1 | 2025-11-16T10:27:51.513631 | None | 2
2 | 015b1d1184288f807143043e2decb8b3 | 2 | 1 | 2025-11-16T10:28:06.306337 | None | 2
3 | 010aa6a0bdfe0194f6ac81b94f4a7cc6 | 2 | 1 | 2025-11-16T10:28:17.196357 | None | 2
4 | 014e6bf3927ddaf505c8053e37c8b688 | 2 | 1 | 2025-11-16T10:28:30.624149 | None | 2
5 | 01c8403779c54eaa3618b44952a05470 | 2 | 1 | 2025-11-16T10:28:41.354496 | None | 2
6 | 016934e7c13b1b546fdea80efd074e02 | 2 | 1 | 2025-11-16T10:28:52.447169 | None | 2
7 | 01ba465f792500066ff5d6bee1f170c1 | 2 | 1 | 2025-11-16T10:29:03.759776 | None | 2
8 | 0819c8effbf081938ef522a369204c9b | 2 | 1 | 2025-11-16T10:29:18.071890 | None | 2
9 | 014e6bf3927ddaf505c8053e37226a1e | 2 | 1 | 2025-11-16T10:29:33.526053 | None | 2
10 | 07d00172c0f2b9dd3facaed885de0c8d | 2 | 1 | 2025-11-16T10:29:45.404114 | None | 2
11 | 0154a0854ad542755c62796906582f56 | 2 | 1 | 2025-11-16

In [ ]:
# SQLite-Verbindung erstellen
conn = sqlite3.connect("../chart_database.db")
cursor = conn.cursor()

cursor.execute(f"SELECT * FROM alt_text;")
data = cursor.fetchall()

# Spaltennamen anzeigen
column_names = [description[0] for description in cursor.description]
print(" | ".join(column_names))  # schöne Formatierung

# Daten anzeigen
for row in data:
    print(" | ".join(str(value) for value in row))

# Verbindung schließen
conn.close()

id | chart_id | generation_run_id | variant_no | short_description_metadata | short_description_overview | long_description
1 | 006f6723bd41797d2bb1e3f65444757f | 1 | 1 | ** Liniendiagramm. Titel: SMI avanciert kräftig. Untertitel: Kursentwicklung in Punkten. Waagrechte Achse: Zeitraum Q1 2017 bis Q4 2017. Senkrechte Achse: Ausschnitt 8000 bis 9600 Punkte. Daten: Täglich Werte. | ** Das ganze Jahr 2017 ist durch einen deutlichen Aufwärtstrend des SMI gekennzeichnet, unterbrochen von Phasen erhöhter Volatilität und Konsolidierung, besonders zwischen dem zweiten und dritten Quartal, bevor der Anstieg im vierten Quartal fortgesetzt wird. | ** Der SMI beginnt das Jahr 2017 bei rund 8316 Punkten, sinkt kurzzeitig auf etwa 8229 (Mitte Januar) und steigt dann stetig. Im April wird die 8800-Punkte-Marke überschritten, worauf eine Phase volatiler Ausschläge folgt, mit einem Hoch von 9177 (Anfang August) und einem Tief von 8814 (Ende August). Ab September setzt sich der Anstieg fort, und das Jah

## Change of ERD and generation_run_table

In [ ]:
import sqlite3

db_path = "../chart_database.db"

conn = sqlite3.connect(db_path)
cur = conn.cursor()

# 1) Get the first ID to keep
cur.execute("SELECT id FROM generation_run ORDER BY id LIMIT 1;")
keep_id = cur.fetchone()[0]

# 2) Delete all other rows
cur.execute("""
DELETE FROM generation_run
WHERE id != ?;
""", (keep_id,))

# 3) Remove column chart_id (SQLite requires table rebuild)
cur.execute("""
CREATE TABLE generation_run_new AS
SELECT 
    id,
    model_id,
    prompt_text_function_id,
    created_at,
    temperature,
    n_variants
FROM generation_run;
""")

cur.execute("DROP TABLE generation_run;")
cur.execute("ALTER TABLE generation_run_new RENAME TO generation_run;")

conn.commit()
conn.close()

In [ ]:
import sqlite3

db_path = "../chart_database.db"

conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute("""
UPDATE alt_text
SET generation_run_id = 1;
""")

conn.commit()
conn.close()

## Check generation_run_table

In [ ]:
# SQLite-Verbindung erstellen
conn = sqlite3.connect("../chart_database.db")
cursor = conn.cursor()

cursor.execute(f"SELECT * FROM generation_run;")
data = cursor.fetchall()

# Spaltennamen anzeigen
column_names = [description[0] for description in cursor.description]
print(" | ".join(column_names))  # schöne Formatierung

# Daten anzeigen
for row in data:
    print(" | ".join(str(value) for value in row))

# Verbindung schließen
conn.close()

id | model_id | prompt_text_function_id | created_at | temperature | n_variants
1 | 2 | 1 | 2025-11-16T10:27:51.513631 | None | 2


## Check alt_text table

In [ ]:
# SQLite-Verbindung erstellen
conn = sqlite3.connect("../chart_database.db")
cursor = conn.cursor()

cursor.execute(f"SELECT * FROM alt_text;")
data = cursor.fetchall()

# Spaltennamen anzeigen
column_names = [description[0] for description in cursor.description]
print(" | ".join(column_names))  # schöne Formatierung

# Daten anzeigen
for row in data:
    print(" | ".join(str(value) for value in row))

# Verbindung schließen
conn.close()

NameError: name 'sqlite3' is not defined